
================================================================

**Author:** Carlos Stalin Saritama Atopo  
**Area of Specialization:** Analytics, Advanced Analytics, and Artificial Intelligence  
**Date:** August 2025 

===============================================================

# 04 — Trading Simulation & Evaluation

**Goals**
- Define strategies using predictions
- Vectorized and iterative backtests
- Portfolio metrics: CAGR, Sharpe, Max Drawdown
- Benchmark comparison

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10,5)

PROC_DIR = Path('../data/processed')
MOD_DIR = Path('../models')
OUT_DIR = Path('../outputs/simulations')
OUT_DIR.mkdir(parents=True, exist_ok=True)

feat = pd.read_parquet(PROC_DIR / 'features.parquet')
preds = pd.read_csv(MOD_DIR / 'predictions.csv')

test_part = feat.sort_values(['ticker','date']).iloc[int(len(feat)*0.8):][['date','ticker','close']].reset_index(drop=True)
df = preds.merge(test_part, on=['date','ticker'], how='left')

df = df.sort_values(['ticker','date'])
df['ret_1d'] = df.groupby('ticker')['close'].pct_change().shift(-1)

sig1 = (df['pred_label']==1).astype(int)
df['sim1_ret'] = sig1 * df['ret_1d']

sig2 = np.where(df['pred_proba']>0.6, 1, np.where(df['pred_proba']<0.4, -1, 0))
df['sim2_ret'] = sig2 * df['ret_1d']

vol = df.groupby('ticker')['ret_1d'].transform(lambda x: x.rolling(20).std())
scale = (vol.mean() / (vol+1e-9)).clip(0,3)
df['sim3_ret'] = np.where(df['pred_proba']>0.55, 1, 0) * df['ret_1d'] * scale

port = df.groupby('date')[['sim1_ret','sim2_ret','sim3_ret','ret_1d']].mean().dropna()
bench = port['ret_1d']

def equity_curve(r):
    return (1+r).cumprod()

def cagr(r, periods_per_year=252):
    T = len(r)/periods_per_year
    return (1+r).prod()**(1/T) - 1

def sharpe(r, rf=0.0, periods_per_year=252):
    ex = r - rf/periods_per_year
    return np.sqrt(periods_per_year) * ex.mean() / (ex.std()+1e-12)

def max_drawdown(ec):
    roll_max = ec.cummax()
    dd = ec/roll_max - 1
    return dd.min()

metrics = {}
for col in ['sim1_ret','sim2_ret','sim3_ret']:
    r = port[col]
    ec = equity_curve(r)
    metrics[col] = {
        'CAGR': cagr(r),
        'Sharpe': sharpe(r),
        'MaxDD': max_drawdown(ec)
    }
bench_metrics = {'CAGR': cagr(bench), 'Sharpe': sharpe(bench), 'MaxDD': max_drawdown(equity_curve(bench))}

metrics, bench_metrics

## Plot Equity Curves

In [ ]:
ec1 = (1+port['sim1_ret']).cumprod()
ec2 = (1+port['sim2_ret']).cumprod()
ec3 = (1+port['sim3_ret']).cumprod()
ecb = (1+bench).cumprod()

plt.plot(ec1, label='Sim1 Long-Only')
plt.plot(ec2, label='Sim2 Long-Short')
plt.plot(ec3, label='Sim3 Risk-Managed')
plt.plot(ecb, label='Benchmark Equal-Weight')
plt.legend(); plt.title('Equity Curves'); plt.show()


## Save Simulation Results

In [ ]:
csv_path = OUT_DIR / 'portfolio_simulations.csv'
port.to_csv(csv_path)
csv_path